# AIC 2026 — Textual KIS tiếng Anh, tiếng Việt và optional reranker

Notebook tái tạo pipeline: dataset audit → exact NumPy index → pinned English control → warm Vietnamese candidate → optional object-aware contrastive reranking.

Trước khi chạy:

1. Chọn **Accelerator = GPU T4** và bật **Internet** trong Kaggle Notebook. Không dùng P100 khi chưa kiểm tra compatibility.
2. Attach dataset chính `doanminhtuan`.
3. Attach source dataset `doanvandat/aic-dataset-kaggle`.
4. Nếu có, attach private Dataset chứa `clip-flat-ip.npz`; notebook chỉ reuse khi metadata, manifest và SHA-256 khớp.
5. Giữ `ENABLE_RERANKING = False` tới khi private labeled benchmark cho phép promote.
6. Chạy lần lượt từ trên xuống. Không sửa `/kaggle/input`.

Source bundle mong đợi: `aic-retrieval-kaggle.zip`; SHA-256 được pin trong cell kế tiếp sau local verification.

## 1. Tìm source mới, copy sang `/kaggle/working`


In [ ]:
from pathlib import Path
import json
import os
import shutil
import sys

INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
OPENAI_REVISION = "3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268"
MULTILINGUAL_REVISION = "58edf8cada9e398793dca955574a48cbb7f18be2"
PLANNER_REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
# Replaced only after deterministic local bundle verification.
EXPECTED_BUNDLE_SHA256 = "dc9ae93704ab9adb15c87ab0ebdf466604d29fb51d7ca45f6b9601e78dc8d0e3"

SOURCE_MOUNTS = [
    INPUT / "datasets" / "doanvandat" / "aic-dataset-kaggle",
    INPUT / "aic-dataset-kaggle",
    INPUT / "doanvandat" / "aic-dataset-kaggle",
]

def is_current_source(root: Path) -> bool:
    query_source = root / "src" / "aic_retrieval" / "query.py"
    reranking_source = root / "src" / "aic_retrieval" / "reranking.py"
    english_config = root / "config" / "query-encoder.example.yaml"
    multilingual_config = root / "config" / "query-encoder-multilingual.example.yaml"
    reranker_config = root / "config" / "reranker-disabled.example.yaml"
    required = [
        root / "pyproject.toml",
        root / "scripts" / "audit_dataset.py",
        root / "scripts" / "search.py",
        query_source,
        reranking_source,
        english_config,
        multilingual_config,
        reranker_config,
    ]
    if not all(path.is_file() for path in required):
        return False
    try:
        english = json.loads(english_config.read_text(encoding="utf-8"))
        multilingual = json.loads(multilingual_config.read_text(encoding="utf-8"))
        reranker = json.loads(reranker_config.read_text(encoding="utf-8"))
        query_content = query_source.read_text(encoding="utf-8").replace(" ", "")
        reranking_content = reranking_source.read_text(encoding="utf-8").replace(" ", "")
    except (OSError, json.JSONDecodeError):
        return False
    return (
        english.get("backend") == "openai-clip"
        and english.get("revision") == OPENAI_REVISION
        and english.get("tokenizer_use_fast") is False
        and multilingual.get("backend") == "sentence-transformers"
        and multilingual.get("revision") == MULTILINGUAL_REVISION
        and reranker.get("enabled") is False
        and reranker.get("planner_revision") == PLANNER_REVISION
        and "use_safetensors=False" in query_content
        and "trust_remote_code=False" in query_content
        and "trust_remote_code=False" in reranking_content
    )

candidates = []
for mount in SOURCE_MOUNTS:
    if not mount.exists():
        continue
    if is_current_source(mount):
        candidates.append(mount)
    for pyproject in mount.rglob("pyproject.toml"):
        root = pyproject.parent
        if is_current_source(root):
            candidates.append(root)

if not candidates:
    visible = [str(path) for path in INPUT.iterdir() if path.is_dir()]
    raise FileNotFoundError(
        "Không tìm thấy source bundle reranker mới. Các mount hiện có:\n"
        + "\n".join(visible)
    )

SOURCE = sorted(set(candidates), key=lambda path: len(path.parts))[0]
PROJECT = WORKING / f"aic-retrieval-{EXPECTED_BUNDLE_SHA256[:12]}"

if PROJECT.exists():
    if not is_current_source(PROJECT):
        raise RuntimeError(
            f"{PROJECT} đã tồn tại nhưng không phải source mới. "
            "Restart session hoặc đổi tên thư mục."
        )
    print("Reuse source:", PROJECT)
else:
    shutil.copytree(
        SOURCE,
        PROJECT,
        ignore=shutil.ignore_patterns(".git", "__pycache__", "*.pyc", "dist"),
    )
    print("Copied source:", SOURCE, "->", PROJECT)

SRC = PROJECT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.environ["PYTHONPATH"] = str(SRC)
ENV = os.environ.copy()

assert is_current_source(PROJECT)
print("PROJECT:", PROJECT)
print("PYTHONPATH:", os.environ["PYTHONPATH"])
print("Expected source ZIP SHA-256:", EXPECTED_BUNDLE_SHA256)
print("English + multilingual + optional reranker source: verified")

## 2. Kiểm tra/cài dependencies

Không cài editable project. Notebook chạy source bằng `PYTHONPATH`. Sentence Transformers chỉ phục vụ Vietnamese candidate; vector mode vẫn NumPy-only.

In [ ]:
import importlib.util
import subprocess

if importlib.util.find_spec("torch") is None:
    raise RuntimeError(
        "PyTorch không có trong runtime. Đổi Kaggle Accelerator sang GPU T4 rồi restart."
    )

missing = []
if importlib.util.find_spec("transformers") is None:
    missing.append("transformers>=4.40")
if importlib.util.find_spec("sentence_transformers") is None:
    missing.append("sentence-transformers>=5.0")
if importlib.util.find_spec("cv2") is None:
    missing.append("opencv-python-headless>=4.8")

if missing:
    print("Installing:", missing)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing],
        check=True,
    )

import cv2
import numpy as np
import sentence_transformers
import torch
import transformers

assert torch.cuda.is_available(), (
    "CUDA chưa hoạt động. Chọn GPU T4 rồi restart session."
)
gpu_name = torch.cuda.get_device_name(0)
assert "T4" in gpu_name.upper(), (
    f"Notebook yêu cầu GPU T4; runtime hiện tại: {gpu_name}. "
    "Đổi Accelerator rồi restart session."
)
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("CUDA:", gpu_name)

## 3. Xác định dataset, tạo runtime config


In [ ]:
DATASET_CANDIDATES = [
    Path("/kaggle/input/datasets/doanminhtuan"),
    Path("/kaggle/input/doanminhtuan"),
]
DATASET = next((path for path in DATASET_CANDIDATES if path.is_dir()), None)
if DATASET is None:
    raise FileNotFoundError(
        "Không tìm thấy dataset doanminhtuan tại:\n"
        + "\n".join(map(str, DATASET_CANDIDATES))
    )

required_dataset_paths = [
    DATASET / "video-aic",
    DATASET / "dataset-ai-challenge-keyframe",
    DATASET / "map-keyframes-aic25-b1",
    DATASET / "clip-features-32-aic25-b1",
    DATASET / "media-info-aic25-b1",
    DATASET / "objects-aic25-b1-zip",
]
missing_paths = [str(path) for path in required_dataset_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Dataset thiếu các thư mục:\n" + "\n".join(missing_paths)
    )

AUDIT_DIR = WORKING / "aic-audit"
INDEX_DIR = WORKING / "aic-index"
RESULT_DIR = WORKING / "aic-results"
for directory in (AUDIT_DIR, INDEX_DIR, RESULT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

AUDIT_CONFIG = WORKING / "dataset.json"
AUDIT_REPORT = AUDIT_DIR / "report.json"
MANIFEST = AUDIT_DIR / "manifest.json"
INDEX = INDEX_DIR / "clip-flat-ip.npz"
INDEX_CONFIG = WORKING / "index.json"
ENGLISH_ENCODER_CONFIG = WORKING / "query-encoder-english.json"
VIETNAMESE_ENCODER_CONFIG = WORKING / "query-encoder-vietnamese.json"
ENGLISH_RESULT = RESULT_DIR / "kis-search-english.json"
VIETNAMESE_RESULT = RESULT_DIR / "kis-search-vietnamese.json"

audit_config = json.loads(
    (PROJECT / "config" / "dataset.example.yaml").read_text(encoding="utf-8")
)
audit_config.update({
    "dataset_root": str(DATASET),
    "output_manifest": str(MANIFEST),
    "output_report": str(AUDIT_REPORT),
    "frame_id_source": "timestamp",
    "verify_frame_mapping": True,
    "fail_on_warning": False,
})
AUDIT_CONFIG.write_text(
    json.dumps(audit_config, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print("DATASET:", DATASET)
print("Audit config:", AUDIT_CONFIG)
print("Manifest:", MANIFEST)
print("Index:", INDEX)

## 4. Audit dataset nếu artifact sạch chưa tồn tại

Full audit có thể chạy lâu. Notebook không audit lại khi report và manifest sạch còn tồn tại.


In [ ]:
EXPECTED_COUNTS = {
    "videos": 873,
    "keyframes": 177321,
    "mappings": 873,
    "clip_arrays": 873,
    "object_files": 177321,
    "metadata_files": 873,
}

def clean_audit_exists() -> bool:
    if not AUDIT_REPORT.is_file() or not MANIFEST.is_file():
        return False
    try:
        existing_report = json.loads(AUDIT_REPORT.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return False
    return (
        existing_report.get("valid") is True
        and existing_report.get("counts") == EXPECTED_COUNTS
        and existing_report.get("issues") == []
    )

if clean_audit_exists():
    print("Clean audit đã tồn tại; bỏ qua audit lại.")
else:
    if AUDIT_REPORT.exists() or MANIFEST.exists():
        raise RuntimeError(
            "Có audit artifact cũ nhưng không clean. Không tự ghi đè. "
            "Kiểm tra report hoặc restart session."
        )
    subprocess.run(
        [
            sys.executable,
            str(PROJECT / "scripts" / "audit_dataset.py"),
            "--config",
            str(AUDIT_CONFIG),
        ],
        cwd=PROJECT,
        env=ENV,
        check=True,
    )


## 5. Xác nhận audit sạch


In [ ]:
import hashlib

report = json.loads(AUDIT_REPORT.read_text(encoding="utf-8"))
assert report["valid"] is True
assert report["counts"] == EXPECTED_COUNTS
assert report["issues"] == []
assert MANIFEST.is_file() and MANIFEST.stat().st_size > 0

manifest_sha256 = hashlib.sha256(MANIFEST.read_bytes()).hexdigest()
print("Audit valid:", report["valid"])
print("Counts:", json.dumps(report["counts"], indent=2))
print("Issues:", len(report["issues"]))
print("Manifest size:", f"{MANIFEST.stat().st_size / 2**20:.2f} MiB")
print("Manifest SHA-256:", manifest_sha256)


## 6. Reuse hoặc tạo exact NumPy index

Notebook ưu tiên private attached `clip-flat-ip.npz`. Chỉ copy sang `/kaggle/working` khi metadata, manifest và known SHA-256 đều khớp. Không sửa `/kaggle/input`; không thêm FAISS.

In [ ]:
index_config = {
    "manifest_path": str(MANIFEST),
    "dataset_root": str(DATASET),
    "output_path": str(INDEX),
    "feature_glob": (
        "clip-features-32-aic25-b1/clip-features-32/{video_id}.npy"
    ),
    "model_id": (
        "provided CLIP features; empirically matched to "
        "openai/clip-vit-base-patch32"
    ),
    "preprocessing": (
        "provided float16 features; empirically matched with "
        "CLIPImageProcessor(use_fast=False); source generator/revision unknown"
    ),
}
INDEX_CONFIG.write_text(
    json.dumps(index_config, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

EXPECTED_INDEX_SHA256 = (
    "df0d4de81b25d198fa6bb605f0cc2a978fccfe61fe75246f34ed81af2ce373b8"
)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def read_index_metadata(path: Path):
    if not path.is_file():
        return None
    try:
        with np.load(path, allow_pickle=False) as archive:
            return json.loads(str(archive["metadata_json"]))
    except Exception:
        return None

def compatible_index_content(path: Path) -> bool:
    metadata = read_index_metadata(path)
    return (
        metadata is not None
        and metadata.get("backend") == "numpy-flat-ip"
        and metadata.get("rows") == 177321
        and metadata.get("dimension") == 512
        and metadata.get("normalized") is True
        and metadata.get("manifest_sha256") == manifest_sha256
    )

def known_index_archive(path: Path) -> bool:
    return (
        compatible_index_content(path)
        and sha256_file(path) == EXPECTED_INDEX_SHA256
    )

if INDEX.exists():
    if not compatible_index_content(INDEX):
        raise RuntimeError("Index working hiện có không khớp manifest/metadata.")
    print("Reuse compatible index trong /kaggle/working.")
else:
    attached = []
    for path in INPUT.rglob("clip-flat-ip.npz"):
        try:
            if known_index_archive(path):
                attached.append(path)
        except (OSError, ValueError):
            continue
    if attached:
        source_index = sorted(attached, key=lambda path: len(path.parts))[0]
        shutil.copyfile(source_index, INDEX)
        print("Copied verified private index:", source_index, "->", INDEX)
    else:
        subprocess.run(
            [
                sys.executable,
                str(PROJECT / "scripts" / "build_index.py"),
                "--config",
                str(INDEX_CONFIG),
            ],
            cwd=PROJECT,
            env=ENV,
            check=True,
        )

## 7. Verify index, ghi checksum


In [ ]:
subprocess.run(
    [
        sys.executable,
        str(PROJECT / "scripts" / "verify_index.py"),
        str(INDEX),
    ],
    cwd=PROJECT,
    env=ENV,
    check=True,
)

with np.load(INDEX, allow_pickle=False) as archive:
    index_metadata = json.loads(str(archive["metadata_json"]))

assert index_metadata["backend"] == "numpy-flat-ip"
assert index_metadata["rows"] == 177321
assert index_metadata["dimension"] == 512
assert index_metadata["normalized"] is True
assert index_metadata["manifest_sha256"] == manifest_sha256

index_sha256 = sha256_file(INDEX)
known_index_archive_match = index_sha256 == EXPECTED_INDEX_SHA256
print(json.dumps(index_metadata, indent=2, ensure_ascii=False))
print("Index size:", f"{INDEX.stat().st_size / 2**20:.2f} MiB")
print("Index SHA-256:", index_sha256)
print("Matches known archive:", known_index_archive_match)
if not known_index_archive_match:
    print(
        "Archive bytes khác bản đã lưu, nhưng verify_index và metadata/manifest "
        "đều hợp lệ; tiếp tục dùng index vừa build."
    )

## 8. Tạo hai pinned query encoder config

English control giữ OpenAI CLIP. Vietnamese candidate chỉ đổi text encoder; exact image index không đổi.

In [ ]:
english_encoder = json.loads(
    (PROJECT / "config" / "query-encoder.example.yaml").read_text(
        encoding="utf-8"
    )
)
vietnamese_encoder = json.loads(
    (PROJECT / "config" / "query-encoder-multilingual.example.yaml").read_text(
        encoding="utf-8"
    )
)
assert english_encoder == {
    "backend": "openai-clip",
    "model_id": "openai/clip-vit-base-patch32",
    "revision": OPENAI_REVISION,
    "tokenizer_use_fast": False,
    "device": "cuda",
}
assert vietnamese_encoder == {
    "backend": "sentence-transformers",
    "model_id": "sentence-transformers/clip-ViT-B-32-multilingual-v1",
    "revision": MULTILINGUAL_REVISION,
    "device": "cuda",
}
ENGLISH_ENCODER_CONFIG.write_text(
    json.dumps(english_encoder, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
VIETNAMESE_ENCODER_CONFIG.write_text(
    json.dumps(vietnamese_encoder, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print(ENGLISH_ENCODER_CONFIG.read_text(encoding="utf-8"))
print(VIETNAMESE_ENCODER_CONFIG.read_text(encoding="utf-8"))

## 9. Xác minh hai Hugging Face checkpoint revision

Cần Internet nếu checkpoint chưa có trong cache. Không cần `HF_TOKEN`; `trust_remote_code=False`.

In [ ]:
from transformers import AutoConfig
from sentence_transformers import SentenceTransformer
import gc

english_hf_config = AutoConfig.from_pretrained(
    "openai/clip-vit-base-patch32",
    revision=OPENAI_REVISION,
)
english_loaded_revision = getattr(english_hf_config, "_commit_hash", None)
assert english_loaded_revision == OPENAI_REVISION, (
    f"English checkpoint {english_loaded_revision!r} != {OPENAI_REVISION!r}"
)

multilingual_probe = SentenceTransformer(
    "sentence-transformers/clip-ViT-B-32-multilingual-v1",
    revision=MULTILINGUAL_REVISION,
    device="cuda",
    trust_remote_code=False,
)
first_module = multilingual_probe._first_module()
auto_model = getattr(first_module, "auto_model", None)
multilingual_hf_config = getattr(
    auto_model,
    "config",
    getattr(first_module, "config", None),
)
multilingual_loaded_revision = getattr(
    multilingual_hf_config,
    "_commit_hash",
    None,
)
if multilingual_loaded_revision is not None:
    assert multilingual_loaded_revision == MULTILINGUAL_REVISION, (
        f"Multilingual checkpoint {multilingual_loaded_revision!r} "
        f"!= {MULTILINGUAL_REVISION!r}"
    )
assert multilingual_probe.get_embedding_dimension() == 512
print("English revision:", english_loaded_revision)
print("Multilingual revision:", multilingual_loaded_revision or "runtime omitted field")
print("Multilingual dimension: 512")
del multilingual_probe, first_module, auto_model, multilingual_hf_config
gc.collect()
torch.cuda.empty_cache()

## 10. Khởi tạo warm runtime; chạy nhiều query

Index và Vietnamese encoder load đúng một lần. `search_query(text)` dùng lại runtime. Reranker mặc định tắt; khi bật chỉ hiển thị aggregate status, không hiển thị internal plan.

In [ ]:
from dataclasses import asdict
import gc
import time

from aic_retrieval.index import load_index
from aic_retrieval.query import QueryEncoderRuntime, load_config as load_encoder_config
from aic_retrieval.reranking import (
    ContrastiveReranker,
    QwenPlanner,
    RerankingError,
    RerankStatus,
    load_config as load_reranker_config,
)
from aic_retrieval.retrieval import load_config as load_retrieval_config, retrieve_kis

ENGLISH_QUERY = "a man riding a motorcycle on a street"
VIETNAMESE_QUERY = "một người đàn ông đang chạy xe máy trên đường phố"
RETRIEVAL_CONFIG = PROJECT / "config" / "retrieval-baseline.yaml"
ENABLE_RERANKING = False
RERANKER_CONFIG = (
    WORKING / "reranker-private.json"
    if ENABLE_RERANKING
    else PROJECT / "config" / "reranker-disabled.example.yaml"
)
RERANKER_STATUS_KEYS = {
    "applied",
    "fallback",
    "circuit_open",
    "planner_elapsed_ms",
    "scoring_elapsed_ms",
}
reranker_startup_status = RerankStatus(False, False, False, 0.0, 0.0)

print("Loading index...", flush=True)
started = time.perf_counter()
index = load_index(INDEX)
print("Index loaded:", f"{time.perf_counter() - started:.2f} s", flush=True)
retrieval_config = load_retrieval_config(RETRIEVAL_CONFIG)

def encode_retrieve(text: str, runtime: QueryEncoderRuntime, reranker=None):
    started = time.perf_counter()
    encoded = runtime.encode([text])
    encoding_ms = (time.perf_counter() - started) * 1000.0
    started = time.perf_counter()
    result = retrieve_kis(
        index,
        encoded.vectors[0],
        retrieval_config,
        rerank=(
            (lambda idx, hits: reranker.rerank(text, idx, hits))
            if reranker is not None
            else None
        ),
    )
    retrieval_ms = (time.perf_counter() - started) * 1000.0
    status = asdict(
        reranker.last_status if reranker is not None else reranker_startup_status
    )
    return encoded, result, encoding_ms, retrieval_ms, status

def write_private_safe_result(path: Path, text: str, payload: dict) -> None:
    content = json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True) + "\n"
    assert text not in content, "Raw query bị serialize"
    temporary = path.with_name(f".{path.name}.tmp")
    temporary.write_text(content, encoding="utf-8")
    temporary.replace(path)

def reusable_english_result_exists() -> bool:
    if not ENGLISH_RESULT.is_file():
        return False
    try:
        content = ENGLISH_RESULT.read_text(encoding="utf-8")
        payload = json.loads(content)
        encoder = payload["encoder"]
        status = payload["reranker"]
    except (OSError, json.JSONDecodeError, KeyError, TypeError):
        return False
    return (
        ENGLISH_QUERY not in content
        and isinstance(encoder, dict)
        and encoder.get("backend") == "openai-clip"
        and encoder.get("revision") == OPENAI_REVISION
        and isinstance(status, dict)
        and set(status) == RERANKER_STATUS_KEYS
    )

# English control remains reproducible, but its runtime is released before warm Vietnamese search.
if reusable_english_result_exists():
    print("Reuse English control result:", ENGLISH_RESULT)
else:
    if ENGLISH_RESULT.exists():
        print("English control artifact dùng schema cũ; rebuild atomically.")
    english_config = load_encoder_config(ENGLISH_ENCODER_CONFIG)
    english_runtime = QueryEncoderRuntime(
        english_config,
        expected_dimension=index.metadata.dimension,
    )
    encoded, result, encoding_ms, retrieval_ms, status = encode_retrieve(
        ENGLISH_QUERY,
        english_runtime,
    )
    english_payload = {
        "index": asdict(index.metadata),
        "config": asdict(retrieval_config),
        "encoder": asdict(encoded.provenance),
        "reranker": status,
        "encoding_elapsed_ms": encoding_ms,
        "retrieval_elapsed_ms": retrieval_ms,
        "elapsed_ms": encoding_ms + retrieval_ms,
        **result.to_dict(),
    }
    write_private_safe_result(ENGLISH_RESULT, ENGLISH_QUERY, english_payload)
    del english_runtime
    gc.collect()
    torch.cuda.empty_cache()

vietnamese_config = load_encoder_config(VIETNAMESE_ENCODER_CONFIG)
vietnamese_runtime = QueryEncoderRuntime(
    vietnamese_config,
    expected_dimension=index.metadata.dimension,
)
reranker_config = load_reranker_config(RERANKER_CONFIG)
assert reranker_config.enabled is ENABLE_RERANKING
reranker = None
if ENABLE_RERANKING:
    try:
        planner = QwenPlanner(
            reranker_config.planner_model_id,
            reranker_config.planner_revision,
            device=vietnamese_config.device,
        )
    except RerankingError:
        reranker_startup_status = RerankStatus(False, True, True, 0.0, 0.0)
    else:
        reranker = ContrastiveReranker(
            reranker_config,
            planner,
            vietnamese_runtime,
            DATASET,
        )

def search_query(text: str, *, output_path: Path | None = None):
    encoded, result, encoding_ms, retrieval_ms, status = encode_retrieve(
        text,
        vietnamese_runtime,
        reranker,
    )
    payload = {
        "index": asdict(index.metadata),
        "config": asdict(retrieval_config),
        "encoder": asdict(encoded.provenance),
        "reranker": status,
        "encoding_elapsed_ms": encoding_ms,
        "retrieval_elapsed_ms": retrieval_ms,
        "elapsed_ms": encoding_ms + retrieval_ms,
        **result.to_dict(),
    }
    if output_path is not None:
        write_private_safe_result(output_path, text, payload)
    return payload

vietnamese_payload = search_query(VIETNAMESE_QUERY, output_path=VIETNAMESE_RESULT)
print("Warm Vietnamese query complete:", f"{vietnamese_payload['elapsed_ms']:.2f} ms")
print("Reranker status:", vietnamese_payload["reranker"])
print("Dùng tiếp: results = search_query(\"query tiếng Việt khác\")")

## 11. Kiểm tra privacy, provenance, latency, top results

Contact sheet chỉ là smoke evidence. Không dùng để promote model; promotion cần private labeled R@k.

In [ ]:
def load_search_result(path: Path, query: str, backend: str, revision: str):
    content = path.read_text(encoding="utf-8")
    payload = json.loads(content)
    assert query not in content, "Raw query bị serialize vào output"
    assert payload["index"]["rows"] == 177321
    assert payload["index"]["dimension"] == 512
    assert payload["encoder"]["backend"] == backend
    assert payload["encoder"]["revision"] == revision
    assert payload["encoder"]["normalized"] is True
    assert payload["encoder"]["output_dtype"] == "float32"
    assert set(payload["reranker"]) == {
        "applied",
        "fallback",
        "circuit_open",
        "planner_elapsed_ms",
        "scoring_elapsed_ms",
    }
    assert len(payload["responses"]) == 100
    assert len({
        (item["video_id"], item["frame_id"])
        for item in payload["responses"]
    }) == 100
    forbidden = (
        "positive",
        "negatives",
        "required_objects",
        "excluded_objects",
        "object_path",
        "detection_scores",
    )
    for field in forbidden:
        assert f'"{field}"' not in content, f"Private reranker field serialized: {field}"
    return content, payload

english_content, english_payload = load_search_result(
    ENGLISH_RESULT,
    ENGLISH_QUERY,
    "openai-clip",
    OPENAI_REVISION,
)
assert english_payload["encoder"]["model_id"] == "openai/clip-vit-base-patch32"
assert english_payload["encoder"]["tokenizer_use_fast"] is False

vietnamese_content, vietnamese_payload = load_search_result(
    VIETNAMESE_RESULT,
    VIETNAMESE_QUERY,
    "sentence-transformers",
    MULTILINGUAL_REVISION,
)
assert vietnamese_payload["encoder"]["model_id"] == (
    "sentence-transformers/clip-ViT-B-32-multilingual-v1"
)
assert vietnamese_payload["encoder"]["runtime_class"] == "SentenceTransformer"
assert vietnamese_payload["encoder"]["tokenizer_use_fast"] is None
if not ENABLE_RERANKING:
    assert vietnamese_payload["reranker"] == {
        "applied": False,
        "fallback": False,
        "circuit_open": False,
        "planner_elapsed_ms": 0.0,
        "scoring_elapsed_ms": 0.0,
    }

for name, payload in (
    ("English control", english_payload),
    ("Vietnamese candidate", vietnamese_payload),
):
    print(f"\n{name}")
    print("Encoding latency:", f"{payload['encoding_elapsed_ms']:.2f} ms")
    print("Retrieval latency:", f"{payload['retrieval_elapsed_ms']:.2f} ms")
    print("Total latency:", f"{payload['elapsed_ms']:.2f} ms")
    print("Reranker:", payload["reranker"])
    print("Top 10:")
    for rank, item in enumerate(payload["final_candidates"][:10], start=1):
        print(
            f"{rank:02d}. {item['video_id']} | keyframe={item['keyframe_id']} | "
            f"frame={item['original_frame_id']} | score={item['score']:.6f}"
        )

## 12. Contact sheets riêng: English control và Vietnamese candidate

Chỉ hiển thị trong notebook; không export sample frames.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def show_contact_sheet(title: str, payload: dict) -> None:
    top = payload["final_candidates"][:10]
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    fig.suptitle(title, fontsize=16)
    for rank, (axis, item) in enumerate(zip(axes.flat, top), start=1):
        image_path = DATASET / item["keyframe_path"]
        with Image.open(image_path) as image:
            axis.imshow(image.convert("RGB"))
        axis.set_title(
            f"{rank}. {item['video_id']}\n"
            f"KF {item['keyframe_id']} | frame {item['original_frame_id']}\n"
            f"score {item['score']:.4f}",
            fontsize=9,
        )
        axis.axis("off")
    plt.tight_layout()
    plt.show()

show_contact_sheet("English control", english_payload)
show_contact_sheet("Vietnamese candidate", vietnamese_payload)

## 13. Tổng hợp artifacts cần giữ


In [ ]:
artifacts = [
    AUDIT_REPORT,
    MANIFEST,
    INDEX,
    ENGLISH_RESULT,
    VIETNAMESE_RESULT,
    AUDIT_CONFIG,
    INDEX_CONFIG,
    ENGLISH_ENCODER_CONFIG,
    VIETNAMESE_ENCODER_CONFIG,
]
for path in artifacts:
    assert path.is_file(), f"Missing artifact: {path}"
    print(f"{path} | {path.stat().st_size / 2**20:.2f} MiB")

summary = {
    "source_bundle_sha256": EXPECTED_BUNDLE_SHA256,
    "encoders": {
        "english_control": {
            "model": english_payload["encoder"]["model_id"],
            "revision": english_payload["encoder"]["revision"],
            "encoding_elapsed_ms": english_payload["encoding_elapsed_ms"],
            "retrieval_elapsed_ms": english_payload["retrieval_elapsed_ms"],
        },
        "vietnamese_candidate": {
            "model": vietnamese_payload["encoder"]["model_id"],
            "revision": vietnamese_payload["encoder"]["revision"],
            "encoding_elapsed_ms": vietnamese_payload["encoding_elapsed_ms"],
            "retrieval_elapsed_ms": vietnamese_payload["retrieval_elapsed_ms"],
        },
    },
    "reranker": vietnamese_payload["reranker"],
    "reranker_enabled": ENABLE_RERANKING,
    "manifest_sha256": manifest_sha256,
    "index_sha256": sha256_file(INDEX),
    "index_rows": 177321,
    "index_dimension": 512,
    "audit_counts": report["counts"],
    "search_results": {
        "english_control": str(ENGLISH_RESULT),
        "vietnamese_candidate": str(VIETNAMESE_RESULT),
    },
    "raw_queries_serialized": {
        "english_control": ENGLISH_QUERY in english_content,
        "vietnamese_candidate": VIETNAMESE_QUERY in vietnamese_content,
    },
    "internal_plan_serialized": any(
        f'"{field}"' in vietnamese_content
        for field in (
            "positive",
            "negatives",
            "required_objects",
            "excluded_objects",
        )
    ),
    "promotion_status": "pending private labeled R@k and T4 ablation benchmark",
}
SUMMARY_PATH = RESULT_DIR / "artifact-summary.json"
SUMMARY_PATH.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print("\nSummary:", SUMMARY_PATH)
print(SUMMARY_PATH.read_text(encoding="utf-8"))

## Hoàn tất

Trước khi restart:

1. Dùng `search_query("query tiếng Việt khác")` để xử lý nhiều query; không chạy lại audit/index/model cells.
2. Giữ `ENABLE_RERANKING = False` tới khi bốn private ablations tăng held-out R@k/Final Score và đạt T4 p95/memory gate.
3. Chọn **Save Version → Save & Run All** để Kaggle lưu notebook outputs.
4. Giữ audit, manifest, exact index, search JSON và aggregate summary trong private Kaggle output Dataset.
5. Không public query labels, raw vectors, internal plans, object metadata, credentials hoặc sample frames.

`/kaggle/working` là tạm thời. Restart/reset xóa warm runtime, circuit breaker và có thể xóa artifacts.